# [3]submit-T5-v1

- 元: `notebooks/008/[2]T5-v1.ipynb`（学習済み T5 モデルを保存済み）
- 参考: `notebooks/002/[4-8]submit-notebook-v1.ipynb`（学習済みモデルで test 推論→submission 作成）

このノートブックは **学習は行わず**、保存済みモデルを使って `test.csv` を推論し、`/kaggle/working/submission.csv` を作ります。

## 使い方

- Kaggle でこのノートブックに「学習済みモデルが入ったフォルダ」を Dataset / Model としてアタッチ
- 必要なら環境変数 `MODEL_DIR` をそのフォルダに設定（例: `/kaggle/input/<your-dataset>/<model-folder>`）

※ scoring error 対策として、提出直前に簡易バリデーションと「空文字→`<gap>`」の後処理を入れています（`[4-8]` と同様）。


In [1]:
import os
import re
import math
import unicodedata
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from IPython.display import display

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import T5Tokenizer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


'false'

In [2]:
@dataclass
class CFG:
    # Competition data path (Kaggle may mount it in either location)
    input_dirs: tuple[str, ...] = (
        "/kaggle/input/competitions/deep-past-initiative-machine-translation",
        "/kaggle/input/deep-past-initiative-machine-translation",
    )

    # Saved model directory (set via env MODEL_DIR if needed)
    # Default assumes you exported artifacts from `[2]T5-v1.ipynb`.
    model_dir: str | None = os.environ.get("MODEL_DIR")
    default_model_dir: str = "/kaggle/input/notebooks/tatsuokoshida/2-t5-v1"

    output_dir: str = "/kaggle/working"

    # Tokenization / generation (match `[2]T5-v1.ipynb`)
    max_length: int = 1024

    # Inference
    batch_size: int = 16
    num_workers: int = 2

    # Decoding
    num_beams: int = 1

    def __post_init__(self):
        # MODEL_DIR が空文字のケースを吸収し、必ず文字列パスへ正規化する。
        if self.model_dir is None or str(self.model_dir).strip() == "":
            self.model_dir = self.default_model_dir
        else:
            self.model_dir = str(self.model_dir).strip()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)


cfg = CFG()
print("PyTorch:", torch.__version__)
print("Device :", cfg.device)
print("MODEL_DIR:", cfg.model_dir)


def resolve_input_dir(input_dirs: tuple[str, ...]) -> str:
    for d in input_dirs:
        if Path(d).exists():
            return d
    raise FileNotFoundError(f"Competition input dir not found. tried={list(input_dirs)}")


def _has_model_weights(dir_path: Path) -> bool:
    # config.json が無くても重みがあれば候補として扱う。
    fixed_files = [
        "pytorch_model.bin",
        "model.safetensors",
        "pytorch_model.bin.index.json",
        "model.safetensors.index.json",
    ]
    if any((dir_path / name).exists() for name in fixed_files):
        return True
    if list(dir_path.glob("pytorch_model-*.bin")):
        return True
    if list(dir_path.glob("model-*.safetensors")):
        return True
    return False


def resolve_model_dir(model_dir: str | None) -> str:
    candidates = [
        model_dir,
        cfg.default_model_dir,
        "/kaggle/working/t5-scratch-spm-p100-medium-akkadian-model",
    ]

    valid_candidates: list[str] = []
    for c in candidates:
        if c is None:
            continue
        s = str(c).strip()
        if s == "":
            continue
        if s not in valid_candidates:
            valid_candidates.append(s)

    if not valid_candidates:
        raise FileNotFoundError("Model directory candidates are empty. Set MODEL_DIR to a valid mounted path.")

    # 1) 直接候補（config または重み）
    for d in valid_candidates:
        p = Path(d)
        if p.is_dir() and ((p / "config.json").exists() or _has_model_weights(p)):
            return str(p)

    # 2) 親ディレクトリ配下を再帰探索
    hits: list[str] = []
    for d in valid_candidates:
        p = Path(d)
        if not p.is_dir():
            continue
        for child in p.rglob("*"):
            if not child.is_dir():
                continue
            if (child / "config.json").exists() or _has_model_weights(child):
                hits.append(str(child))

    # 一意ならそのまま採用、複数なら最短パスを採用（通常は model 直下が最短）
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        hits = sorted(set(hits), key=lambda x: (len(x), x))
        print("[WARN] Multiple model-like directories found. Using:", hits[0])
        print("[WARN] Candidates:", hits[:5])
        return hits[0]

    raise FileNotFoundError(
        "Model directory not found. Set MODEL_DIR to a mounted path containing saved model artifacts. "
        f"Tried: {valid_candidates}"
    )


PyTorch: 2.9.0+cu126
Device : cuda


In [3]:
input_dir = resolve_input_dir(cfg.input_dirs)

test_path = f"{input_dir}/test.csv"
sample_sub_path = f"{input_dir}/sample_submission.csv"

print("Test  :", test_path)
print("Sample:", sample_sub_path)

test_df = pd.read_csv(test_path)
print("Test rows:", len(test_df))
print("Columns :", list(test_df.columns))

sample_sub_df = None
if Path(sample_sub_path).exists():
    sample_sub_df = pd.read_csv(sample_sub_path)
    print("Sample submission rows:", len(sample_sub_df))
    print("Sample columns        :", list(sample_sub_df.columns))


Test  : /kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv
Sample: /kaggle/input/competitions/deep-past-initiative-machine-translation/sample_submission.csv
Test rows: 4
Columns : ['id', 'text_id', 'line_start', 'line_end', 'transliteration']
Sample submission rows: 4
Sample columns        : ['id', 'translation']


In [4]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
from typing import List

SRC_COL = "transliteration"
TGT_COL = "translation"

PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

# --- Pre/Post processing (aligned to notebooks/006/lb-35-9-ensembling-post-processing-baseline.ipynb) ---
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s


# Unicode fraction map (all single-character Unicode vulgar fractions)
_UNICODE_FRAC_MAP = {
    (1, 2): "½", (1, 3): "⅓", (2, 3): "⅔",
    (1, 4): "¼", (3, 4): "¾",
    (1, 5): "⅕", (2, 5): "⅖", (3, 5): "⅗", (4, 5): "⅘",
    (1, 6): "⅙", (5, 6): "⅚",
    (1, 7): "⅐",
    (1, 8): "⅛", (3, 8): "⅜", (5, 8): "⅝", (7, 8): "⅞",
    (1, 9): "⅑",
    (1, 10): "⅒",
}
def _trunc_float(x: float, places: int = 4) -> float:
    factor = 10 ** places
    if x >= 0:
        return math.floor(x * factor) / factor
    return -math.floor(-x * factor) / factor

def _fmt_trunc(x: float, places: int = 4) -> str:
    return f"{_trunc_float(x, places):.{places}f}".rstrip("0").rstrip(".")

# Map the 4-decimal *truncated* representation to a Unicode fraction.
# (Host example: 1.333330000... -> 1.3333; 0.1666 -> ⅙)
_FRAC_DECIMAL_MAP_4 = {}
for (n, d), ch in _UNICODE_FRAC_MAP.items():
    _FRAC_DECIMAL_MAP_4[_fmt_trunc(n / d, 4)] = ch

# Decimals (incl. long floats)
_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d+)(?![\w/])")
_LONG_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d{5,})(?![\w/])")

# General fractions like "1/2" or mixed fractions like "2 1/2".
_MIXED_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s+(\d+)\s*/\s*(\d+)(?![\w/])")
_SIMPLE_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s*/\s*(\d+)(?![\w/])")

def _mixed_frac_repl(m: re.Match) -> str:
    ip = int(m.group(1))
    num = int(m.group(2))
    den = int(m.group(3))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return f"{ip} {ch}" if ip != 0 else ch
    return m.group(0)

def _simple_frac_repl(m: re.Match) -> str:
    num = int(m.group(1))
    den = int(m.group(2))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return ch
    return m.group(0)

def _canon_decimal_str(s: str) -> str:
    # Keep x.5 (e.g., 2.5) as-is by request.
    if re.fullmatch(r"[1-9]\d*\.5", s):
        return s

    m = re.fullmatch(r"(\d+)\.(\d+)", s)
    if not m:
        return s

    ip = int(m.group(1))
    frac_digits = m.group(2)

    # Truncate fractional part to 4 digits (no rounding), pad with zeros for lookup.
    frac4 = (frac_digits + "0000")[:4]
    frac_key = ("0." + frac4).rstrip("0").rstrip(".")
    ch = _FRAC_DECIMAL_MAP_4.get(frac_key)
    if ch and frac_key != "0":
        if ip == 0:
            return ch
        return f"{ip} {ch}"

    # Long-float shortening: truncate to 4 digits after the decimal.
    if len(frac_digits) > 4:
        return f"{ip}.{frac4}".rstrip("0").rstrip(".")

    return s


_TAG_BIGGAP_RE = re.compile(r"<\s*big[\s_\-]*gap\s*>", re.I)
_TAG_GAP_RE    = re.compile(r"<\s*gap\s*>", re.I)
_BARE_BIGGAP   = re.compile(r"\bbig[\s_\-]*gap\b", re.I)
_ELLIPSIS_RE   = re.compile(r"(?:\.{3,}|…+|\[\.+\])")
_BRACKET_X_RE  = re.compile(r"(\[\s*x\s*\]|\(\s*x\s*\))", re.I)
_XTOKEN_RUN_RE = re.compile(r"\bx(?:\s+x)+\b", re.I)
_XRUN_RE       = re.compile(r"(?<!\w)x{2,}(?!\w)", re.I)
_XTOK_RE       = re.compile(r"(?<!\w)x(?!\w)", re.I)
_WS_RE         = re.compile(r"\s+")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)


_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_PN_RE = re.compile(r"\bPN\b")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")


class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)

        # Uppercase determinatives are unwrapped, lowercase ones are converted to braces.
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)

        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        ser = ser.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        ser = ser.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()

_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)", re.I
)
_BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE = re.compile(r"\(\?\)")
_CURLY_QUOTES_RE = re.compile("[\u201c\u201d\u2018\u2019]")

_MONTH_RE = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}

_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE = re.compile(r"\s+([.,:])")

_FORBIDDEN_TRANS = str.maketrans("", "", '()——<>⌈⌋⌊[]+ʾ;')

_COMMODITY_RE = re.compile(r'-(gold|tax|textiles)\b')
_COMMODITY_REPL = {
    "gold": "pašallum gold",
    "tax": "šadduātum tax",
    "textiles": "kutānum textiles",
}

def _commodity_repl(m: re.Match) -> str:
    return _COMMODITY_REPL[m.group(1)]


_SHEKEL_REPLS = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?', re.I), '⅔ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?', re.I), '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]

_ALT_SLASH_PAIR_RE = re.compile(r"\b([^\W\d_]+)\s*/\s*([^\W\d_]+)\b")
_STRAY_MARKS_RE = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP_RE = re.compile(r'(?:<gap>\s*){2,}')

def _month_repl(m: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"


class VectorizedPostprocessor:
    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations).fillna("").astype(str)

        s = _normalize_gaps_vec(s)
        s = s.str.replace(_PN_RE, "<gap>", regex=True)
        s = s.str.replace(_COMMODITY_RE, _commodity_repl, regex=True)

        for pat, repl in _SHEKEL_REPLS:
            s = s.str.replace(pat, repl, regex=True)

        s = s.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        s = s.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        s = s.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)

        s = s.str.replace(_SOFT_GRAM_RE, " ", regex=True)
        s = s.str.replace(_BARE_GRAM_RE, " ", regex=True)
        s = s.str.replace(_UNCERTAIN_RE, "", regex=True)

        s = s.str.replace(_STRAY_MARKS_RE, "", regex=True)
        # Keep the left alternative (e.g., "you / she" -> "you")
        s = s.str.replace(_ALT_SLASH_PAIR_RE, r"\1", regex=True)

        # Remove only curly quotes; keep straight quotes and apostrophes.
        s = s.str.replace(_CURLY_QUOTES_RE, "", regex=True)

        s = s.str.replace(_MONTH_RE, _month_repl, regex=True)
        s = s.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)

        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)

        s = s.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)
        for n in range(4, 1, -1):
            pat = r"\b((?:\w+\s+){" + str(n-1) + r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pat, r"\1", regex=True)

        s = s.str.replace(_PUNCT_SPACE_RE, r"\1", regex=True)
        s = s.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
        s = s.str.replace(_WS_RE, " ", regex=True).str.strip()

        return s.tolist()

_preprocessor = OptimizedPreprocessor()
_postprocessor = VectorizedPostprocessor()




def clean_transliteration(s: str) -> str:
    return _preprocessor.preprocess_batch([s])[0]


def clean_translation(s: str) -> str:
    return _postprocessor.postprocess_batch([s])[0]


def postprocess_translation(text: str) -> str:
    return clean_translation(text)


In [5]:
model_dir = resolve_model_dir(cfg.model_dir)
print("Model:", model_dir)

# Tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
except Exception as e:
    print("AutoTokenizer failed. fallback to T5Tokenizer. err=", repr(e))
    tokenizer = T5Tokenizer.from_pretrained(model_dir, local_files_only=True)


def _load_state_dict_for_shape_inference(model_dir_path: Path):
    # config 再生成に必要な shape 情報だけを得るために state_dict を読む。
    import json as _json

    safetensors_file = model_dir_path / "model.safetensors"
    bin_file = model_dir_path / "pytorch_model.bin"

    if safetensors_file.exists():
        from safetensors.torch import load_file as safe_load_file
        return safe_load_file(str(safetensors_file), device="cpu")

    if bin_file.exists():
        return torch.load(str(bin_file), map_location="cpu")

    safetensors_index = model_dir_path / "model.safetensors.index.json"
    if safetensors_index.exists():
        from safetensors.torch import load_file as safe_load_file
        weight_map = _json.loads(safetensors_index.read_text())["weight_map"]
        needed_keys = [
            "shared.weight",
            "encoder.block.0.layer.1.DenseReluDense.wi.weight",
            "encoder.block.0.layer.1.DenseReluDense.wi_0.weight",
        ]
        needed_files = {weight_map[k] for k in needed_keys if k in weight_map}
        if not needed_files:
            needed_files = set(weight_map.values())
        merged = {}
        for f in sorted(needed_files):
            merged.update(safe_load_file(str(model_dir_path / f), device="cpu"))
        return merged

    bin_index = model_dir_path / "pytorch_model.bin.index.json"
    if bin_index.exists():
        weight_map = _json.loads(bin_index.read_text())["weight_map"]
        needed_keys = [
            "shared.weight",
            "encoder.block.0.layer.1.DenseReluDense.wi.weight",
            "encoder.block.0.layer.1.DenseReluDense.wi_0.weight",
        ]
        needed_files = {weight_map[k] for k in needed_keys if k in weight_map}
        if not needed_files:
            needed_files = set(weight_map.values())
        merged = {}
        for f in sorted(needed_files):
            merged.update(torch.load(str(model_dir_path / f), map_location="cpu"))
        return merged

    raise FileNotFoundError(
        "No model weights found. Expected one of: model.safetensors / pytorch_model.bin / sharded index files"
    )


def _count_blocks(state_dict: dict, prefix: str) -> int:
    idx = set()
    for k in state_dict.keys():
        if not k.startswith(prefix):
            continue
        parts = k.split(".")
        if len(parts) > 2 and parts[2].isdigit():
            idx.add(int(parts[2]))
    return (max(idx) + 1) if idx else 0


def ensure_t5_config_json(model_dir_str: str, tokenizer_obj) -> Path:
    from transformers import T5Config

    model_path = Path(model_dir_str)
    config_path = model_path / "config.json"
    if config_path.exists():
        return config_path

    print("[INFO] config.json が見つからないため、重みから再構成します。")
    state_dict = _load_state_dict_for_shape_inference(model_path)

    if "shared.weight" not in state_dict:
        raise ValueError("Cannot infer config because 'shared.weight' is missing in model weights.")

    vocab_size, d_model = state_dict["shared.weight"].shape

    wi_key = None
    feed_forward_proj = "relu"
    if "encoder.block.0.layer.1.DenseReluDense.wi.weight" in state_dict:
        wi_key = "encoder.block.0.layer.1.DenseReluDense.wi.weight"
    elif "encoder.block.0.layer.1.DenseReluDense.wi_0.weight" in state_dict:
        wi_key = "encoder.block.0.layer.1.DenseReluDense.wi_0.weight"
        feed_forward_proj = "gated-gelu"
    else:
        raise ValueError("Cannot infer d_ff because wi.weight / wi_0.weight is missing.")

    d_ff = int(state_dict[wi_key].shape[0])
    num_layers = _count_blocks(state_dict, "encoder.block.")
    num_decoder_layers = _count_blocks(state_dict, "decoder.block.")

    # [2]T5-v1 で使っている preset に合わせた既定値。
    heads_by_d_model = {256: 8, 512: 8, 768: 12}
    num_heads = heads_by_d_model.get(int(d_model), 8)

    cfg_obj = T5Config(
        vocab_size=int(vocab_size),
        d_model=int(d_model),
        d_ff=int(d_ff),
        num_layers=int(num_layers) if num_layers > 0 else 12,
        num_decoder_layers=int(num_decoder_layers) if num_decoder_layers > 0 else 12,
        num_heads=int(num_heads),
        dropout_rate=0.1,
        pad_token_id=int(getattr(tokenizer_obj, "pad_token_id", 0) or 0),
        eos_token_id=int(getattr(tokenizer_obj, "eos_token_id", 1) or 1),
        decoder_start_token_id=int(getattr(tokenizer_obj, "pad_token_id", 0) or 0),
        feed_forward_proj=feed_forward_proj,
    )
    cfg_obj.save_pretrained(str(model_path))
    print(f"[INFO] Reconstructed config.json at: {config_path}")
    return config_path


_ = ensure_t5_config_json(model_dir, tokenizer)

# Model
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir, local_files_only=True)
model.to(cfg.device)
model.eval()

# Make sure generation length matches training/inference defaults
try:
    model.generation_config.max_length = cfg.max_length
except Exception:
    pass


TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'NoneType'

In [ ]:
SRC_COL = "transliteration"
assert SRC_COL in test_df.columns, f"test_df must contain '{SRC_COL}'. got={list(test_df.columns)}"

MAX_LENGTH = cfg.max_length

class TestInferenceDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer):
        texts = df[SRC_COL].fillna("").astype(str).tolist()
        texts = _preprocessor.preprocess_batch(texts)
        self.texts = [PREFIX_AKK_EN + t for t in texts]
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx: int):
        encoded = self.tokenizer(
            self.texts[idx],
            max_length=MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
        }


test_dataset = TestInferenceDataset(test_df, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

preds: list[str] = []
print("Start inference...")
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(cfg.device)
        attention_mask = batch["attention_mask"].to(cfg.device)

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=MAX_LENGTH,
            do_sample=False,
            num_beams=cfg.num_beams,
        )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        decoded = [postprocess_translation(x).strip() for x in decoded]
        preds.extend(decoded)

print("Preds:", len(preds))
print("Example preds:")
display(pd.DataFrame({"pred": preds[:5]}))


In [ ]:
SUBMISSION_PATH = Path(cfg.output_dir) / "submission.csv"

# Build base submission DF
if sample_sub_df is not None:
    submission_df = sample_sub_df.copy()
    id_col = submission_df.columns[0]
    target_col = submission_df.columns[1]
else:
    # Fallback (should not happen in Kaggle competition environment)
    id_col = "id" if "id" in test_df.columns else test_df.columns[0]
    target_col = "translation"
    submission_df = pd.DataFrame({id_col: test_df[id_col].tolist()})

# Length safety (avoid ValueError like Length of values != index)
if len(preds) != len(submission_df):
    print(f"[WARN] len(preds)={len(preds)} != len(submission_df)={len(submission_df)}. Fixing...")
    if len(preds) > len(submission_df):
        preds = preds[: len(submission_df)]
    else:
        preds = preds + [""] * (len(submission_df) - len(preds))

submission_df[target_col] = preds

# scoring error 対策（空文字→<gap> + validate）
submission_df[target_col] = submission_df[target_col].fillna("").astype(str)
submission_df.loc[submission_df[target_col].str.strip() == "", target_col] = "<gap>"


def validate_submission(df: pd.DataFrame, expected_rows: int):
    expected_columns = [id_col, target_col]
    if list(df.columns) != expected_columns:
        raise ValueError(f"Submission columns must be {expected_columns}. got={list(df.columns)}")
    if len(df) != expected_rows:
        raise ValueError(f"Submission rows mismatch. expected={expected_rows}, got={len(df)}")
    if df[id_col].isna().any():
        raise ValueError("Submission contains missing id values.")
    if df[id_col].duplicated().any():
        dup_ids = df.loc[df[id_col].duplicated(), id_col].astype(str).head(5).tolist()
        raise ValueError(f"Submission contains duplicated id values. examples={dup_ids}")
    empty_mask = df[target_col].fillna("").astype(str).str.strip() == ""
    if empty_mask.any():
        bad_ids = df.loc[empty_mask, id_col].astype(str).head(5).tolist()
        raise ValueError(f"Submission contains empty translations. examples={bad_ids}")


validate_submission(submission_df, expected_rows=len(submission_df))
submission_df.to_csv(SUBMISSION_PATH, index=False)

print("Saved to:", str(SUBMISSION_PATH))
print("Submission preview:")
display(submission_df.head())
